# Notebook 02 — Data Preprocessing

**Chapter 3, Section 3.5.2 — Data Preprocessing Pipeline**

## Pipeline steps demonstrated:
1. Drop irrelevant columns (`difficulty`, `_split`)
2. Handle missing and infinite values (mean/mode imputation)
3. Remove duplicate rows
4. Map raw attack-type labels → 5-class integers (0–4)
5. One-hot encode categorical features (`protocol_type`, `service`, `flag`)
6. Min-Max scale all numeric features to [0, 1]
7. Save interim DataFrames + preprocessing artifacts

---

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.helpers import set_global_seed, print_banner
from src.config import get_config

cfg = get_config()
set_global_seed(cfg.seed)
print_banner('Notebook 02 — Data Preprocessing')

## 1. Load Raw Data

In [ ]:
from src.data.loaders import load_dataset

main_df, _ = load_dataset('nsl_kdd', merge=True, validate=True)
print(f'Merged dataset: {main_df.shape}')
print(f'Label column sample: {main_df["label"].unique()[:8]}')

## 2. Step 1 — Drop Irrelevant Columns

In [ ]:
from src.data.preprocessing import drop_irrelevant_columns

print(f'Columns before: {len(main_df.columns)}')
df = drop_irrelevant_columns(main_df.copy(), dataset='nsl_kdd')
print(f'Columns after : {len(df.columns)}')
print(f'Dropped: difficulty, _split')

## 3. Step 2 — Handle Missing & Infinite Values

In [ ]:
from src.data.preprocessing import handle_missing_and_infinite

missing_before = df.isnull().sum().sum()
df = handle_missing_and_infinite(
    df,
    strategy_continuous=cfg.preprocessing.missing_strategy_continuous,
    strategy_categorical=cfg.preprocessing.missing_strategy_categorical
)
missing_after = df.isnull().sum().sum()
print(f'Missing values: {missing_before:,} → {missing_after:,}')

## 4. Step 3 — Remove Duplicates

In [ ]:
from src.data.preprocessing import remove_duplicates

n_before = len(df)
df = remove_duplicates(df)
print(f'Rows: {n_before:,} → {len(df):,} (removed {n_before - len(df):,} duplicates)')

## 5. Step 4 — Map Labels to Integer Classes

In [ ]:
from src.data.preprocessing import map_nsl_kdd_labels
from src.utils.constants import NSL_KDD_CLASS_NAMES

df = map_nsl_kdd_labels(df)

dist = df['label'].value_counts().sort_index()
print('Label distribution after mapping:')
for cls_int, cnt in dist.items():
    name = NSL_KDD_CLASS_NAMES[cls_int]
    print(f'  {cls_int} ({name:<10}) : {cnt:>8,} ({cnt/len(df)*100:.2f}%)')

## 6. Step 5 — One-Hot Encoding

In [ ]:
from src.data.preprocessing import encode_categorical_features

print(f'Columns before encoding: {len(df.columns)}')
df_encoded = encode_categorical_features(df.copy(), dataset='nsl_kdd', drop_first=False)
print(f'Columns after encoding : {len(df_encoded.columns)}')

# Show which one-hot columns were created
ohe_cols = [c for c in df_encoded.columns if
            any(c.startswith(cat) for cat in ['protocol_type_', 'service_', 'flag_'])]
print(f'\nNew OHE columns ({len(ohe_cols)} total): {ohe_cols[:8]} ...')

## 7. Step 6 — Separate Features and Target

In [ ]:
from src.data.preprocessing import separate_features_target

X_df, y = separate_features_target(df_encoded, dataset='nsl_kdd')
feature_names = X_df.columns.tolist()

print(f'Feature matrix X: {X_df.shape}')
print(f'Target vector  y: {y.shape}')
print(f'Number of features after encoding: {len(feature_names)}')

## 8. Step 7 — Min-Max Scaling

In [ ]:
from src.data.preprocessing import fit_scaler, apply_scaler

scaler = fit_scaler(X_df, feature_range=(0.0, 1.0))
X_scaled = apply_scaler(X_df, scaler)

print(f'Scaled X shape  : {X_scaled.shape}')
print(f'Scaled X min    : {X_scaled.min():.6f}')
print(f'Scaled X max    : {X_scaled.max():.6f}')
print(f'All values in [0,1]: {(X_scaled.min() >= -0.001) and (X_scaled.max() <= 1.001)}')

## 9. Full Pipeline (One-Call Shortcut)

In [ ]:
from src.data.preprocessing import preprocess_dataset
from src.utils.paths import PROCESSED_DATA_DIR, FINAL_MODEL_DIR

main_df2, _ = load_dataset('nsl_kdd', merge=True, validate=False)

X_scaled_full, y_full, scaler_full, feature_names_full, metadata = \
    preprocess_dataset(
        main_df2,
        dataset='nsl_kdd',
        save_interim_files=True,
        artifacts_dir=PROCESSED_DATA_DIR,
    )

print('Preprocessing pipeline complete!')
print(f'  X shape      : {X_scaled_full.shape}')
print(f'  y shape      : {y_full.shape}')
print(f'  n_features   : {metadata["n_features"]}')
print(f'  n_classes    : {metadata["n_classes"]}')
print(f'  class names  : {metadata["class_names"]}')

In [ ]:
# Verify class distribution
import numpy as np
unique, counts = np.unique(y_full, return_counts=True)
print('Final class distribution:')
for cls, cnt in zip(unique, counts):
    name = metadata['class_names'][cls]
    print(f'  {cls} ({name:<10}) : {cnt:>8,} ({cnt/len(y_full)*100:.2f}%)')

## 10. Save Artifacts

In [ ]:
from src.utils.serialization import save_preprocessing_artifacts
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.classes_ = np.array(metadata['class_names'])

save_preprocessing_artifacts(
    scaler=scaler_full,
    label_encoder=le,
    feature_names=feature_names_full,
    metadata=metadata,
    output_dir=FINAL_MODEL_DIR,
)
print(f'Artifacts saved to: {FINAL_MODEL_DIR}')

---
**Summary:** The preprocessing pipeline has been successfully applied.  
Interim CSVs saved to `data/interim/` and preprocessing artifacts to `data/processed/` and `models/final/`.  
**Next:** Run `03_sequence_generation.ipynb`